In [4]:
import json
from pathlib import Path

In [10]:
data = []

with open("mpchat_hybrid_sentiment_updated.json", "r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if not line:
            continue
        data.append(json.loads(line))

print(f"Loaded {len(data)} samples")


Loaded 7007 samples


In [11]:
import pprint

pprint.pprint(data[0])


{'context': [{'debug': {'alice_sentiment': 0.7177,
                        'neg_score': 0.5,
                        'pos_score': 1.0,
                        'similarity': 0.35043150186538696},
              'sentiment_towards_target': 'positive',
              'speaker_id': 'stylofylo',
              'turn_index': 1,
              'utterance': 'omg i work at the cafe where we sell ur pins! '
                           "they're so cute"}],
 'dialog_id': '6ee8654c-4189-49bf-b272-6c6f9a1cb99d',
 's_node': {'speaker_id': 'nickeymousee',
            'turn_index': 2,
            'utterance': 'which cafe? i only have them at trueshot right now '
                         'and in edmonton soon edit: saw a comment for '
                         'hamilton haha, so cool'},
 'target_speaker': 'nickeymousee'}


In [12]:
def build_interaction_text(sample):
    """
    Converts one dataset sample into a clean, persona-focused interaction text.
    """
    ctx = sample["context"][-1]  # recency already enforced upstream
    s = sample["s_node"]

    interaction = f"""
Other user ({ctx['speaker_id']}) said
(sentiment toward {sample['target_speaker']}: {ctx['sentiment_towards_target']}):
"{ctx['utterance']}"

{sample['target_speaker']} replied:
"{s['utterance']}"
""".strip()

    return interaction


In [13]:
print(build_interaction_text(data[0]))

Other user (stylofylo) said
(sentiment toward nickeymousee: positive):
"omg i work at the cafe where we sell ur pins! they're so cute"

nickeymousee replied:
"which cafe? i only have them at trueshot right now and in edmonton soon edit: saw a comment for hamilton haha, so cool"


In [14]:
def build_memory_prompt(speaker, previous_memory, interaction_text):
    """
    Builds the prompt used to recursively update persona memory.
    """
    return f"""
You are maintaining a long-term memory about the speaker "{speaker}".

Current memory:
{previous_memory if previous_memory else "None"}

New interaction:
{interaction_text}

Update the memory to include any new information about the speaker's:
- personality
- preferences
- opinions
- recurring behavior

Rules:
- Do NOT repeat dialogue verbatim
- Do NOT mention other users unless relevant
- Keep the memory concise and factual
""".strip()


In [15]:
prompt = build_memory_prompt(
    speaker=data[0]["target_speaker"],
    previous_memory="",
    interaction_text=build_interaction_text(data[0])
)

print(prompt)


You are maintaining a long-term memory about the speaker "nickeymousee".

Current memory:
None

New interaction:
Other user (stylofylo) said
(sentiment toward nickeymousee: positive):
"omg i work at the cafe where we sell ur pins! they're so cute"

nickeymousee replied:
"which cafe? i only have them at trueshot right now and in edmonton soon edit: saw a comment for hamilton haha, so cool"

Update the memory to include any new information about the speaker's:
- personality
- preferences
- opinions
- recurring behavior

Rules:
- Do NOT repeat dialogue verbatim
- Do NOT mention other users unless relevant
- Keep the memory concise and factual


In [16]:
def call_llm(prompt):
    """
    Temporary stub for LLM call.
    Replace this with a real LLM later.
    """
    # For now, just return a deterministic placeholder
    return "The speaker appears friendly and engages positively with others about their work."


In [17]:
from collections import defaultdict

def run_recursive_summarization(dataset):
    memories = defaultdict(str)   # memory per target_speaker
    augmented_dataset = []

    for idx, sample in enumerate(dataset):
        speaker = sample["target_speaker"]

        interaction_text = build_interaction_text(sample)

        prompt = build_memory_prompt(
            speaker=speaker,
            previous_memory=memories[speaker],
            interaction_text=interaction_text
        )

        updated_memory = call_llm(prompt)

        # Update memory store
        memories[speaker] = updated_memory

        # Attach memory to sample
        new_sample = sample.copy()
        new_sample["memory_summary"] = updated_memory

        augmented_dataset.append(new_sample)

    return augmented_dataset, memories


In [18]:
test_data = data[:5]

augmented_data, final_memories = run_recursive_summarization(test_data)

for d in augmented_data:
    print("=" * 70)
    print("Target speaker:", d["target_speaker"])
    print("Memory summary:", d["memory_summary"])


Target speaker: nickeymousee
Memory summary: The speaker appears friendly and engages positively with others about their work.
Target speaker: Soggypaper_
Memory summary: The speaker appears friendly and engages positively with others about their work.
Target speaker: Soggypaper_
Memory summary: The speaker appears friendly and engages positively with others about their work.
Target speaker: Soggypaper_
Memory summary: The speaker appears friendly and engages positively with others about their work.
Target speaker: BeastaBubbles
Memory summary: The speaker appears friendly and engages positively with others about their work.


# Memory Length & Schema Enforcement

In [24]:
def clamp_text(text, max_sentences=3):
    """
    Ensures memory remains short and summary-like.
    """
    if not text:
        return ""

    sentences = [s.strip() for s in text.split(".") if s.strip()]
    return ". ".join(sentences[:max_sentences])

In [25]:
def stabilize_memory(prev, new):
    """
    Avoid unnecessary memory churn.
    """
    if not new:
        return prev
    if new.strip() == prev.strip():
        return prev
    return new

In [26]:
from collections import defaultdict

def init_memory():
    return {
        "facts": "",
        "style": ""
    }

memories = defaultdict(init_memory)

In [27]:
def run_recursive_summarization(dataset):
    memories = defaultdict(init_memory)
    augmented_dataset = []

    for sample in dataset:
        speaker = sample["target_speaker"]
        interaction_text = build_interaction_text(sample)

        # --- STUB OUTPUT (for now) ---
        raw_memory_update = {
            "facts": "The speaker sells products at cafes and mentions locations.",
            "style": "The speaker responds in a friendly and humorous manner."
        }

        # --- Stabilize + clamp ---
        for k in raw_memory_update:
            raw_memory_update[k] = clamp_text(raw_memory_update[k])
            raw_memory_update[k] = stabilize_memory(
                memories[speaker][k],
                raw_memory_update[k]
            )

        memories[speaker] = raw_memory_update

        # Attach structured memory
        new_sample = sample.copy()
        new_sample["memory_summary"] = memories[speaker]

        augmented_dataset.append(new_sample)

    return augmented_dataset, memories

In [28]:
test_data = data[:3]
augmented_data, final_memories = run_recursive_summarization(test_data)

for d in augmented_data:
    print("=" * 60)
    print("Speaker:", d["target_speaker"])
    print("Memory:", d["memory_summary"])


Speaker: nickeymousee
Memory: {'facts': 'The speaker sells products at cafes and mentions locations', 'style': 'The speaker responds in a friendly and humorous manner'}
Speaker: Soggypaper_
Memory: {'facts': 'The speaker sells products at cafes and mentions locations', 'style': 'The speaker responds in a friendly and humorous manner'}
Speaker: Soggypaper_
Memory: {'facts': 'The speaker sells products at cafes and mentions locations', 'style': 'The speaker responds in a friendly and humorous manner'}


# Speaker-wise Chronological Ordering

In [29]:
from collections import defaultdict

def group_by_speaker(dataset):
    grouped = defaultdict(list)
    for sample in dataset:
        grouped[sample["target_speaker"]].append(sample)
    return grouped


In [30]:
def sort_speaker_samples(grouped_samples):
    for speaker in grouped_samples:
        grouped_samples[speaker].sort(
            key=lambda x: x["s_node"]["turn_index"]
        )
    return grouped_samples


In [31]:
def flatten_grouped_samples(grouped_samples):
    ordered_dataset = []
    for speaker in grouped_samples:
        ordered_dataset.extend(grouped_samples[speaker])
    return ordered_dataset


In [32]:
def reorder_dataset_by_speaker_time(dataset):
    grouped = group_by_speaker(dataset)
    grouped = sort_speaker_samples(grouped)
    return flatten_grouped_samples(grouped)


In [33]:
ordered_data = reorder_dataset_by_speaker_time(data)

test_data = ordered_data[:5]
augmented_data, final_memories = run_recursive_summarization(test_data)

for d in augmented_data:
    print("=" * 60)
    print("Speaker:", d["target_speaker"])
    print("Turn index:", d["s_node"]["turn_index"])
    print("Memory:", d["memory_summary"])


Speaker: nickeymousee
Turn index: 2
Memory: {'facts': 'The speaker sells products at cafes and mentions locations', 'style': 'The speaker responds in a friendly and humorous manner'}
Speaker: nickeymousee
Turn index: 2
Memory: {'facts': 'The speaker sells products at cafes and mentions locations', 'style': 'The speaker responds in a friendly and humorous manner'}
Speaker: nickeymousee
Turn index: 2
Memory: {'facts': 'The speaker sells products at cafes and mentions locations', 'style': 'The speaker responds in a friendly and humorous manner'}
Speaker: nickeymousee
Turn index: 2
Memory: {'facts': 'The speaker sells products at cafes and mentions locations', 'style': 'The speaker responds in a friendly and humorous manner'}
Speaker: nickeymousee
Turn index: 4
Memory: {'facts': 'The speaker sells products at cafes and mentions locations', 'style': 'The speaker responds in a friendly and humorous manner'}


# Persist the Augmented Dataset

In [35]:
import json

def save_jsonl(dataset, path):
    """
    Saves a list of dicts as JSON Lines (JSONL).
    """
    with open(path, "w", encoding="utf-8") as f:
        for sample in dataset:
            f.write(json.dumps(sample, ensure_ascii=False) + "\n")


In [36]:
# 1. Reorder dataset by speaker/time
ordered_data = reorder_dataset_by_speaker_time(data)

# 2. Run recursive summarization
augmented_data, final_memories = run_recursive_summarization(ordered_data)

# 3. Save to disk
output_path = "mpchat_recursive_memory_stub.jsonl"
save_jsonl(augmented_data, output_path)

print(f"Saved {len(augmented_data)} samples to {output_path}")


Saved 7007 samples to mpchat_recursive_memory_stub.jsonl
